# 05 — Feature Engineering

This notebook converts the preprocessed respiratory-cycle audio into
structured numerical acoustic features.

Each row in the resulting feature dataset represents one respiratory cycle.

The feature-engineering pipeline is motivated by the respiratory sound
classification literature and combines:

1. Time-domain features
2. Frequency-domain features
3. Spectral features
4. Frequency-band energy
5. MFCC features
6. Wavelet-based features

The extracted features will later be used for:

- exploratory feature analysis
- correlation analysis
- feature selection
- PCA
- clustering
- classical machine-learning models

The target labels and patient metadata are retained separately from the
acoustic features.

No machine-learning model is trained in this notebook.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import scipy.stats as stats
import pywt

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

FEATURES_DIR = (
    PROCESSED_DATA_DIR / "features"
)

FEATURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)
print("Features directory:", FEATURES_DIR)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Processed data: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed
Features directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/features


In [ ]:
processed_cycles_df = pd.read_csv(
    PROCESSED_DATA_DIR / "processed_cycles.csv"
)

print("Processed cycles:", processed_cycles_df.shape)

display(
    processed_cycles_df.head()
)

Processed cycles: (6898, 23)


,patient_id,diagnosis,recording_id,chest_location,acquisition_mode,equipment,cycle_number,sound_label,crackles,wheezes,...,target_sample_rate,original_samples,processed_samples,original_duration,processed_duration,original_peak,original_rms,normalized_peak,processed_path,processing_status
0,101,URTI,1b1,Al,sc,Meditron,1,Normal,0,0,...,4000,23946,2172,0.542993,0.54300,0.357483,0.079829,1.0,audio/cycles/101/1b1_Al_001.wav,success
1,101,URTI,1b1,Al,sc,Meditron,2,Normal,0,0,...,4000,82511,7484,1.870998,1.87100,0.394714,0.073536,1.0,audio/cycles/101/1b1_Al_002.wav,success
2,101,URTI,1b1,Al,sc,Meditron,3,Normal,0,0,...,4000,63636,5772,1.442993,1.44300,0.411743,0.070142,1.0,audio/cycles/101/1b1_Al_003.wav,success
3,101,URTI,1b1,Al,sc,Meditron,4,Normal,0,0,...,4000,83790,7600,1.900000,1.90000,0.414764,0.064327,1.0,audio/cycles/101/1b1_Al_004.wav,success
4,101,URTI,1b1,Al,sc,Meditron,5,Normal,0,0,...,4000,76205,6913,1.728005,1.72825,0.293762,0.057634,1.0,audio/cycles/101/1b1_Al_005.wav,success


In [ ]:
print(
    "Target sample rates:"
)

display(
    processed_cycles_df[
        "target_sample_rate"
    ].value_counts()
)

Target sample rates:


target_sample_rate
4000    6898
Name: count, dtype: int64

In [ ]:
print(
    "Processing status:"
)

display(
    processed_cycles_df[
        "processing_status"
    ].value_counts()
)

Processing status:


processing_status
success    6898
Name: count, dtype: int64

In [ ]:
sample_row = processed_cycles_df.iloc[0]

sample_path = (
    PROCESSED_DATA_DIR /
    sample_row["processed_path"]
)

sample_audio, sample_sr = librosa.load(
    sample_path,
    sr=None,
    mono=True
)

print("Patient:", sample_row["patient_id"])
print("Recording:", sample_row["recording_id"])
print("Chest location:", sample_row["chest_location"])
print("Cycle:", sample_row["cycle_number"])
print("Sound label:", sample_row["sound_label"])
print("Diagnosis:", sample_row["diagnosis"])

print("\nAudio path:", sample_path)
print("Sampling rate:", sample_sr)
print("Samples:", len(sample_audio))
print(
    "Duration:",
    len(sample_audio) / sample_sr
)
print("Minimum:", sample_audio.min())
print("Maximum:", sample_audio.max())

Patient: 101
Recording: 1b1
Chest location: Al
Cycle: 1
Sound label: Normal
Diagnosis: URTI

Audio path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_001.wav
Sampling rate: 4000
Samples: 2172
Duration: 0.543
Minimum: -0.48672485
Maximum: 0.9999695


In [ ]:
def extract_time_domain_features(signal):
    """
    Extract basic time-domain features from a respiratory-cycle waveform.
    """

    signal = np.asarray(signal, dtype=np.float64)

    features = {}

    features["mean"] = np.mean(signal)
    features["std"] = np.std(signal)
    features["variance"] = np.var(signal)

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    features["max"] = np.max(signal)
    features["min"] = np.min(signal)

    features["peak_to_peak"] = (
        np.ptp(signal)
    )

    features["median"] = np.median(signal)

    features["skewness"] = stats.skew(signal)
    features["kurtosis"] = stats.kurtosis(signal)

    zcr = librosa.feature.zero_crossing_rate(
        signal
    )

    features["zero_crossing_rate"] = (
        float(zcr.mean())
    )

    features["energy"] = np.sum(
        signal ** 2
    )

    return features

In [ ]:
time_features = extract_time_domain_features(
    sample_audio
)

time_features

{'mean': np.float64(-0.00850962714376151),
 'std': np.float64(0.22340295936401894),
 'variance': np.float64(0.0499088822526015),
 'rms': np.float64(0.22356497043751586),
 'max': np.float64(0.999969482421875),
 'min': np.float64(-0.486724853515625),
 'peak_to_peak': np.float64(1.4866943359375),
 'median': np.float64(-0.0164642333984375),
 'skewness': np.float64(0.6984393202181433),
 'kurtosis': np.float64(2.107791186278991),
 'zero_crossing_rate': 0.00859375,
 'energy': np.float64(108.55937492661178)}

In [ ]:
def extract_spectral_features(signal, sr):
    """
    Extract frequency-domain and spectral features.
    """

    signal = np.asarray(signal, dtype=np.float64)

    features = {}

    # Short-time Fourier transform
    stft = librosa.stft(
        signal,
        n_fft=512,
        hop_length=128
    )

    magnitude = np.abs(stft)

    # Power spectrum
    power = magnitude ** 2

    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(
        S=magnitude,
        sr=sr
    )

    features["spectral_centroid"] = (
        float(centroid.mean())
    )

    # Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(
        S=magnitude,
        sr=sr
    )

    features["spectral_bandwidth"] = (
        float(bandwidth.mean())
    )

    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(
        S=magnitude,
        sr=sr,
        roll_percent=0.85
    )

    features["spectral_rolloff"] = (
        float(rolloff.mean())
    )

    # Spectral flatness
    flatness = librosa.feature.spectral_flatness(
        S=magnitude
    )

    features["spectral_flatness"] = (
        float(flatness.mean())
    )

    # Total spectral energy
    features["spectral_energy"] = (
        float(np.sum(power))
    )

    # Dominant frequency
    frequencies = librosa.fft_frequencies(
        sr=sr,
        n_fft=512
    )

    mean_spectrum = magnitude.mean(axis=1)

    dominant_idx = np.argmax(
        mean_spectrum
    )

    features["dominant_frequency"] = (
        float(frequencies[dominant_idx])
    )

    # Spectral entropy
    normalized_power = power / (
        np.sum(power, axis=0, keepdims=True)
        + 1e-12
    )

    spectral_entropy = -np.sum(
        normalized_power *
        np.log2(normalized_power + 1e-12),
        axis=0
    )

    features["spectral_entropy"] = (
        float(np.mean(spectral_entropy))
    )

    return features

In [ ]:
spectral_features = extract_spectral_features(
    sample_audio,
    sample_sr
)

spectral_features

{'spectral_centroid': 85.29120411189281,
 'spectral_bandwidth': 172.46767923255516,
 'spectral_rolloff': 150.27573529411765,
 'spectral_flatness': 0.0009554472007213666,
 'spectral_energy': 42084.37111814546,
 'dominant_frequency': 15.625,
 'spectral_entropy': 2.3794182363203973}

In [ ]:
def extract_band_energy_features(signal, sr):
    """
    Calculate normalized energy in four frequency bands.

    Bands:
        0–500 Hz
        500–1000 Hz
        1000–1500 Hz
        1500–2000 Hz
    """

    signal = np.asarray(signal, dtype=np.float64)

    spectrum = np.abs(
        np.fft.rfft(signal)
    ) ** 2

    frequencies = np.fft.rfftfreq(
        len(signal),
        d=1 / sr
    )

    bands = {
        "band_energy_0_500": (0, 500),
        "band_energy_500_1000": (500, 1000),
        "band_energy_1000_1500": (1000, 1500),
        "band_energy_1500_2000": (1500, 2000),
    }

    total_energy = np.sum(spectrum) + 1e-12

    features = {}

    for name, (low, high) in bands.items():

        mask = (
            (frequencies >= low) &
            (frequencies < high)
        )

        band_energy = np.sum(
            spectrum[mask]
        )

        features[name] = (
            band_energy / total_energy
        )

    return features

In [ ]:
band_features = extract_band_energy_features(
    sample_audio,
    sample_sr
)

band_features

{'band_energy_0_500': np.float64(0.9998530176939681),
 'band_energy_500_1000': np.float64(5.362373972994425e-05),
 'band_energy_1000_1500': np.float64(3.477716314755198e-05),
 'band_energy_1500_2000': np.float64(5.831145156670899e-05)}

In [ ]:
def extract_mfcc_features(signal, sr, n_mfcc=13):
    """
    Extract 13 MFCC coefficients and summarize
    each coefficient using its temporal mean.
    """

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=512,
        hop_length=128
    )

    features = {}

    for i in range(n_mfcc):

        features[
            f"mfcc_{i + 1}_mean"
        ] = float(
            np.mean(mfcc[i])
        )

    return features

In [ ]:
mfcc_features = extract_mfcc_features(
    sample_audio,
    sample_sr
)

mfcc_features

{'mfcc_1_mean': -437.9565734863281,
 'mfcc_2_mean': 145.9639892578125,
 'mfcc_3_mean': 62.398502349853516,
 'mfcc_4_mean': 38.109188079833984,
 'mfcc_5_mean': 24.845794677734375,
 'mfcc_6_mean': 30.785465240478516,
 'mfcc_7_mean': 14.649715423583984,
 'mfcc_8_mean': 13.271879196166992,
 'mfcc_9_mean': 15.322942733764648,
 'mfcc_10_mean': 4.408390045166016,
 'mfcc_11_mean': 9.810835838317871,
 'mfcc_12_mean': 11.595977783203125,
 'mfcc_13_mean': 6.33247709274292}

In [ ]:
def extract_wavelet_features(signal):
    """
    Extract compact wavelet-based features.
    """

    signal = np.asarray(
        signal,
        dtype=np.float64
    )

    coefficients = pywt.wavedec(
        signal,
        wavelet="db4",
        level=4
    )

    energies = np.array([
        np.sum(coeff ** 2)
        for coeff in coefficients
    ])

    total_energy = (
        np.sum(energies) + 1e-12
    )

    normalized_energy = (
        energies / total_energy
    )

    wavelet_entropy = -np.sum(
        normalized_energy *
        np.log2(
            normalized_energy + 1e-12
        )
    )

    return {
        "wavelet_energy": float(
            total_energy
        ),
        "wavelet_entropy": float(
            wavelet_entropy
        ),
        "wavelet_std": float(
            np.std(
                np.concatenate(coefficients)
            )
        )
    }

In [ ]:
wavelet_features = extract_wavelet_features(
    sample_audio
)

wavelet_features

{'wavelet_energy': 110.760925728797,
 'wavelet_entropy': 0.06277677218736658,
 'wavelet_std': 0.22444923217558535}

In [ ]:
def extract_all_features(signal, sr):
    """
    Extract the complete handcrafted feature vector
    for one respiratory cycle.
    """

    features = {}

    features.update(
        extract_time_domain_features(signal)
    )

    features.update(
        extract_spectral_features(
            signal,
            sr
        )
    )

    features.update(
        extract_band_energy_features(
            signal,
            sr
        )
    )

    features.update(
        extract_mfcc_features(
            signal,
            sr
        )
    )

    features.update(
        extract_wavelet_features(
            signal
        )
    )

    return features

In [ ]:
sample_features = extract_all_features(
    sample_audio,
    sample_sr
)

print(
    "Number of extracted features:",
    len(sample_features)
)

display(
    pd.Series(sample_features)
)

Number of extracted features: 39


mean                        -0.008510
std                          0.223403
variance                     0.049909
rms                          0.223565
max                          0.999969
min                         -0.486725
peak_to_peak                 1.486694
median                      -0.016464
skewness                     0.698439
kurtosis                     2.107791
zero_crossing_rate           0.008594
energy                     108.559375
spectral_centroid           85.291204
spectral_bandwidth         172.467679
spectral_rolloff           150.275735
spectral_flatness            0.000955
spectral_energy          42084.371118
dominant_frequency          15.625000
spectral_entropy             2.379418
band_energy_0_500            0.999853
band_energy_500_1000         0.000054
band_energy_1000_1500        0.000035
band_energy_1500_2000        0.000058
mfcc_1_mean               -437.956573
mfcc_2_mean                145.963989
mfcc_3_mean                 62.398502
mfcc_4_mean 

In [ ]:
sample_feature_df = pd.DataFrame(
    [sample_features]
)

print(
    "Feature matrix shape:",
    sample_feature_df.shape
)

print(
    "Missing values:",
    sample_feature_df.isnull().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        sample_feature_df.values
    ).sum()
)

Feature matrix shape: (1, 39)
Missing values: 0
Infinite values: 0


In [ ]:
print("Feature names:")

for i, feature_name in enumerate(
    sample_features.keys(),
    start=1
):
    print(
        f"{i:02d}. {feature_name}"
    )

Feature names:
01. mean
02. std
03. variance
04. rms
05. max
06. min
07. peak_to_peak
08. median
09. skewness
10. kurtosis
11. zero_crossing_rate
12. energy
13. spectral_centroid
14. spectral_bandwidth
15. spectral_rolloff
16. spectral_flatness
17. spectral_energy
18. dominant_frequency
19. spectral_entropy
20. band_energy_0_500
21. band_energy_500_1000
22. band_energy_1000_1500
23. band_energy_1500_2000
24. mfcc_1_mean
25. mfcc_2_mean
26. mfcc_3_mean
27. mfcc_4_mean
28. mfcc_5_mean
29. mfcc_6_mean
30. mfcc_7_mean
31. mfcc_8_mean
32. mfcc_9_mean
33. mfcc_10_mean
34. mfcc_11_mean
35. mfcc_12_mean
36. mfcc_13_mean
37. wavelet_energy
38. wavelet_entropy
39. wavelet_std


In [ ]:
print("Number of extracted features:", len(sample_features))

Number of extracted features: 39


In [ ]:
print(
    "Missing values:",
    sample_feature_df.isnull().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(sample_feature_df.values).sum()
)

Missing values: 0
Infinite values: 0


In [ ]:
from tqdm.auto import tqdm

feature_records = []
feature_errors = []

print("Starting feature extraction...")

Starting feature extraction...


In [9]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import scipy.stats as stats
import pywt

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DATA_DIR / "features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DATA_DIR)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Processed data directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed


In [10]:
processed_cycles_df = pd.read_csv(
    PROCESSED_DATA_DIR / "processed_cycles.csv"
)

print("Shape:", processed_cycles_df.shape)
print("\nColumns:")
print(processed_cycles_df.columns.tolist())

Shape: (6898, 23)

Columns:
['patient_id', 'diagnosis', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'cycle_number', 'sound_label', 'crackles', 'wheezes', 'start_time', 'end_time', 'original_sample_rate', 'target_sample_rate', 'original_samples', 'processed_samples', 'original_duration', 'processed_duration', 'original_peak', 'original_rms', 'normalized_peak', 'processed_path', 'processing_status']


In [13]:
def extract_time_domain_features(signal):
    signal = np.asarray(signal, dtype=np.float64)

    features = {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "variance": np.var(signal),
        "rms": np.sqrt(np.mean(signal ** 2)),
        "max": np.max(signal),
        "min": np.min(signal),
        "peak_to_peak": np.ptp(signal),
        "median": np.median(signal),
        "skewness": stats.skew(signal),
        "kurtosis": stats.kurtosis(signal),
        "energy": np.sum(signal ** 2),
    }

    zcr = librosa.feature.zero_crossing_rate(signal)
    features["zero_crossing_rate"] = float(zcr.mean())

    return features


def extract_spectral_features(signal, sr):
    signal = np.asarray(signal, dtype=np.float64)

    stft = librosa.stft(
        signal,
        n_fft=512,
        hop_length=128
    )

    magnitude = np.abs(stft)
    power = magnitude ** 2

    features = {}

    centroid = librosa.feature.spectral_centroid(
        S=magnitude,
        sr=sr
    )
    features["spectral_centroid"] = float(centroid.mean())

    bandwidth = librosa.feature.spectral_bandwidth(
        S=magnitude,
        sr=sr
    )
    features["spectral_bandwidth"] = float(bandwidth.mean())

    rolloff = librosa.feature.spectral_rolloff(
        S=magnitude,
        sr=sr,
        roll_percent=0.85
    )
    features["spectral_rolloff"] = float(rolloff.mean())

    flatness = librosa.feature.spectral_flatness(
        S=magnitude
    )
    features["spectral_flatness"] = float(flatness.mean())

    features["spectral_energy"] = float(np.sum(power))

    frequencies = librosa.fft_frequencies(
        sr=sr,
        n_fft=512
    )

    mean_spectrum = magnitude.mean(axis=1)
    dominant_idx = np.argmax(mean_spectrum)

    features["dominant_frequency"] = float(
        frequencies[dominant_idx]
    )

    normalized_power = power / (
        np.sum(power, axis=0, keepdims=True) + 1e-12
    )

    spectral_entropy = -np.sum(
        normalized_power *
        np.log2(normalized_power + 1e-12),
        axis=0
    )

    features["spectral_entropy"] = float(
        np.mean(spectral_entropy)
    )

    return features


def extract_band_energy_features(signal, sr):
    signal = np.asarray(signal, dtype=np.float64)

    spectrum = np.abs(
        np.fft.rfft(signal)
    ) ** 2

    frequencies = np.fft.rfftfreq(
        len(signal),
        d=1 / sr
    )

    bands = {
        "band_energy_0_500": (0, 500),
        "band_energy_500_1000": (500, 1000),
        "band_energy_1000_1500": (1000, 1500),
        "band_energy_1500_2000": (1500, 2000),
    }

    total_energy = np.sum(spectrum) + 1e-12

    features = {}

    for name, (low, high) in bands.items():
        mask = (
            (frequencies >= low) &
            (frequencies < high)
        )

        band_energy = np.sum(
            spectrum[mask]
        )

        features[name] = float(
            band_energy / total_energy
        )

    return features


def extract_mfcc_features(signal, sr, n_mfcc=13):
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=512,
        hop_length=128
    )

    features = {}

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = float(
            np.mean(mfcc[i])
        )

    return features


def extract_wavelet_features(signal):
    signal = np.asarray(signal, dtype=np.float64)

    coefficients = pywt.wavedec(
        signal,
        wavelet="db4",
        level=4
    )

    energies = np.array([
        np.sum(coeff ** 2)
        for coeff in coefficients
    ])

    total_energy = np.sum(energies) + 1e-12

    normalized_energy = (
        energies / total_energy
    )

    wavelet_entropy = -np.sum(
        normalized_energy *
        np.log2(normalized_energy + 1e-12)
    )

    return {
        "wavelet_energy": float(total_energy),
        "wavelet_entropy": float(wavelet_entropy),
        "wavelet_std": float(
            np.std(np.concatenate(coefficients))
        )
    }


def extract_all_features(signal, sr):
    features = {}

    features.update(
        extract_time_domain_features(signal)
    )

    features.update(
        extract_spectral_features(signal, sr)
    )

    features.update(
        extract_band_energy_features(signal, sr)
    )

    features.update(
        extract_mfcc_features(signal, sr)
    )

    features.update(
        extract_wavelet_features(signal)
    )

    return features


print("All feature extraction functions defined successfully.")

All feature extraction functions defined successfully.


In [14]:
print(extract_all_features)

<function extract_all_features at 0x118a1c1a0>


In [15]:
row = processed_cycles_df.iloc[0]

audio_path = PROCESSED_DATA_DIR / row["processed_path"]

signal, sr = librosa.load(
    audio_path,
    sr=None,
    mono=True
)

features = extract_all_features(signal, sr)

print("Sample rate:", sr)
print("Number of features:", len(features))
print("Missing values:", sum(pd.isna(v) for v in features.values()))
print("Infinite values:", sum(np.isinf(v) for v in features.values()))

Sample rate: 4000
Number of features: 39
Missing values: 0
Infinite values: 0


In [16]:
# ============================================================
# Full Feature Extraction - Notebook 05
# ============================================================

feature_records = []
extraction_errors = []

total_cycles = len(processed_cycles_df)

print(f"Starting feature extraction for {total_cycles} cycles...")
print("=" * 60)

for idx, row in processed_cycles_df.iterrows():

    try:
        # ----------------------------------------------------
        # Construct processed audio path
        # ----------------------------------------------------
        audio_path = (
            PROCESSED_DATA_DIR /
            row["processed_path"]
        )

        # ----------------------------------------------------
        # Load processed audio
        # ----------------------------------------------------
        signal, sr = librosa.load(
            audio_path,
            sr=None,
            mono=True
        )

        # ----------------------------------------------------
        # Extract 39 handcrafted features
        # ----------------------------------------------------
        features = extract_all_features(
            signal,
            sr
        )

        # ----------------------------------------------------
        # Store metadata
        # ----------------------------------------------------
        feature_record = {
            "patient_id": row["patient_id"],
            "recording_id": row["recording_id"],
            "chest_location": row["chest_location"],
            "acquisition_mode": row["acquisition_mode"],
            "equipment": row["equipment"],
            "cycle_number": row["cycle_number"],
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "sound_label": row["sound_label"],
            "crackles": row["crackles"],
            "wheezes": row["wheezes"],
            "diagnosis": row["diagnosis"],
            "processed_path": row["processed_path"],
        }

        # ----------------------------------------------------
        # Add extracted features
        # ----------------------------------------------------
        feature_record.update(features)

        feature_records.append(feature_record)

    except Exception as e:

        extraction_errors.append({
            "row_index": idx,
            "patient_id": row.get("patient_id"),
            "recording_id": row.get("recording_id"),
            "cycle_number": row.get("cycle_number"),
            "processed_path": row.get("processed_path"),
            "error_type": type(e).__name__,
            "error_message": str(e),
        })

    # --------------------------------------------------------
    # Progress display
    # --------------------------------------------------------
    if (idx + 1) % 500 == 0 or (idx + 1) == total_cycles:
        print(
            f"Processed {idx + 1}/{total_cycles} | "
            f"Successful: {len(feature_records)} | "
            f"Failed: {len(extraction_errors)}"
        )

print("=" * 60)
print("Feature extraction complete.")
print(f"Successful: {len(feature_records)}")
print(f"Failed: {len(extraction_errors)}")

Starting feature extraction for 6898 cycles...
Processed 500/6898 | Successful: 500 | Failed: 0
Processed 1000/6898 | Successful: 1000 | Failed: 0
Processed 1500/6898 | Successful: 1500 | Failed: 0
Processed 2000/6898 | Successful: 2000 | Failed: 0
Processed 2500/6898 | Successful: 2500 | Failed: 0
Processed 3000/6898 | Successful: 3000 | Failed: 0
Processed 3500/6898 | Successful: 3500 | Failed: 0
Processed 4000/6898 | Successful: 4000 | Failed: 0
Processed 4500/6898 | Successful: 4500 | Failed: 0
Processed 5000/6898 | Successful: 5000 | Failed: 0
Processed 5500/6898 | Successful: 5500 | Failed: 0
Processed 6000/6898 | Successful: 6000 | Failed: 0
Processed 6500/6898 | Successful: 6500 | Failed: 0
Processed 6898/6898 | Successful: 6898 | Failed: 0
Feature extraction complete.
Successful: 6898
Failed: 0


In [17]:
features_df = pd.DataFrame(feature_records)

print("Feature DataFrame shape:", features_df.shape)

print("\nFirst 5 rows:")
display(features_df.head())

Feature DataFrame shape: (6898, 52)

First 5 rows:


,patient_id,recording_id,chest_location,acquisition_mode,equipment,cycle_number,start_time,end_time,sound_label,crackles,...,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,mfcc_11_mean,mfcc_12_mean,mfcc_13_mean,wavelet_energy,wavelet_entropy,wavelet_std
0,101,1b1,Al,sc,Meditron,1,0.036,0.579,Normal,0,...,14.649715,13.271879,15.322943,4.408390,9.810836,11.595978,6.332477,110.760926,0.062777,0.224449
1,101,1b1,Al,sc,Meditron,2,0.579,2.450,Normal,0,...,11.704497,17.665747,14.362599,8.428231,8.835912,10.312641,6.039036,263.467394,0.090006,0.187299
2,101,1b1,Al,sc,Meditron,3,2.450,3.893,Normal,0,...,9.776377,21.063862,10.441285,5.727220,12.893122,8.313774,3.833181,169.927915,0.150584,0.171195
3,101,1b1,Al,sc,Meditron,4,3.893,5.793,Normal,0,...,12.218080,18.453773,11.239029,5.239138,7.796915,10.357517,4.856807,189.956857,0.142102,0.157822
4,101,1b1,Al,sc,Meditron,5,5.793,7.521,Normal,0,...,11.927266,16.875360,13.980614,4.852463,6.865896,11.010699,6.107927,268.586226,0.154456,0.196740


In [18]:
# Feature columns
feature_columns = [
    col for col in features_df.columns
    if col not in [
        "patient_id",
        "recording_id",
        "chest_location",
        "acquisition_mode",
        "equipment",
        "cycle_number",
        "start_time",
        "end_time",
        "sound_label",
        "crackles",
        "wheezes",
        "diagnosis",
        "processed_path",
    ]
]

print("Number of feature columns:", len(feature_columns))

print("\nMissing values:")
print(features_df[feature_columns].isna().sum().sum())

print("\nInfinite values:")
print(
    np.isinf(
        features_df[feature_columns].to_numpy()
    ).sum()
)

Number of feature columns: 39

Missing values:
0

Infinite values:
0


In [19]:
cycle_id_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "cycle_number"
]

duplicate_count = features_df.duplicated(
    subset=cycle_id_columns
).sum()

print("Duplicate cycle IDs:", duplicate_count)

Duplicate cycle IDs: 0


In [20]:
print("Disease distribution:")
display(
    features_df["diagnosis"]
    .value_counts()
)

print("\nSound-label distribution:")
display(
    features_df["sound_label"]
    .value_counts()
)

Disease distribution:


diagnosis
COPD              5746
Healthy            322
Pneumonia          285
URTI               243
Bronchiolitis      160
Bronchiectasis     104
LRTI                32
Asthma               6
Name: count, dtype: int64


Sound-label distribution:


sound_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506
Name: count, dtype: int64

In [21]:
FEATURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

features_output_path = (
    FEATURES_DIR /
    "respiratory_cycle_features.csv"
)

errors_output_path = (
    FEATURES_DIR /
    "feature_extraction_errors.csv"
)

features_df.to_csv(
    features_output_path,
    index=False
)

pd.DataFrame(extraction_errors).to_csv(
    errors_output_path,
    index=False
)

print("Features saved to:")
print(features_output_path)

print("\nErrors saved to:")
print(errors_output_path)

Features saved to:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/features/respiratory_cycle_features.csv

Errors saved to:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/features/feature_extraction_errors.csv
